# Setting up and running the calculation:

In [1]:
import os # OS routines for NT or Posix depending on what system we're on.

work_directory = "../calculations/task 1"
os.makedirs(work_directory, exist_ok=True)
print("Folder created (if it didn't exist already)")

Folder created (if it didn't exist already)


In [2]:
import shutil # Utility functions for copying and archiving files and directory trees.

source_file = "../src/silicon/pw.scf.silicon.in"
shutil.copy(source_file, work_directory)
print(f"File copied to {work_directory}")

File copied to ../calculations/task 1


In [ ]:
import re # Regular expression operations.  This module provides regular expression matching operations similar to those found in Perl.
import os # OS routines for NT or Posix depending on what system we're on.

def update_variable_in_file(file_path: str, var_name: str, new_value):
    r"""
    Open `file_path` and look for a top–level assignment to `var_name`.
    The first line matching `^var_name\s*=` is replaced so that the
    right‑hand side becomes `repr(new_value)`.  Returns True if a line
    was changed, False otherwise.
    """
    # Pattern that matches the variable name even if there's leading whitespace
    pattern = re.compile(rf'^(\s*{re.escape(var_name)}\s*=\s*).*$')
    updated = False
    with open(file_path, 'r') as f:
        lines = f.readlines()
    with open(file_path, 'w') as f:
        for line in lines:
            m = pattern.match(line)
            if m:
                # m.group(1) captures everything from the start through the = sign
                f.write(f"{m.group(1)}{repr(new_value)}\n")
                updated = True
            else:
                f.write(line)

    if updated:
        print("Variable 'pseudo_dir' updated successfully.")
    else:
        print("Variable 'pseudo_dir' not found or not updated.")

    return updated

input_file = os.path.join(work_directory, "pw.scf.silicon.in")
pseudo_dir = "../../src/pseudos"

_ = update_variable_in_file(input_file, "pseudo_dir", pseudo_dir)

Variable 'pseudo_dir' updated successfully.


In [ ]:
import subprocess # Subprocesses allow you to spawn new processes, connect to their input/output/error pipes, and obtain their return codes.

output_file = os.path.join(work_directory, "pw.scf.silicon.out")

program = "/home/flaviano/Nextcloud/codes/qe-7.5/build/bin/pw.x"

# run pw.x from the work directory, feeding it the input file and
# writing stdout to the designated output file
with open(input_file, 'r') as inp, open(output_file, 'w') as out:
    ret = subprocess.run([program], stdin=inp, stdout=out, cwd=work_directory)

print(f"pw.x exited with return code {ret.returncode}")

pw.x exited with return code 0


# Printing the results:

In [29]:
# Open the output file pw.scf.silicon.out and print energies, exchange-correlation, k-points, Kohn-Sham states
output_file = os.path.join(work_directory, "pw.scf.silicon.out")

# create function that reads a file, search for a keyword and print a block of lines from num_lines_before to num_lines_after.
def print_block_around_keyword(file_path: str, keyword: str, num_lines_before: int = 0, num_lines_after: int = 5):
    with open(file_path, 'r') as f:
        lines = f.readlines()

    for i, line in enumerate(lines):
        if keyword in line:
            start = max(0, i - num_lines_before)
            end = min(len(lines), i + num_lines_after + 1)
            print("".join(lines[start:end]))
            print("-" * 40)  # Separator between blocks

print_block_around_keyword(output_file, "!    total energy ")
print_block_around_keyword(output_file, "Exchange-correlation", num_lines_before=0, num_lines_after=1)
print_block_around_keyword(output_file, "number of k points", num_lines_before=0, num_lines_after=18)
print_block_around_keyword(output_file, "Kohn-Sham states", num_lines_before=1, num_lines_after=1)
print_block_around_keyword(output_file, "G-vector sticks info", num_lines_before=0, num_lines_after=3)

!    total energy              =     -15.85039422 Ry
     estimated scf accuracy    <       0.00000053 Ry

     The total energy is the sum of the following terms:
     one-electron contribution =       4.65269432 Ry
     hartree contribution      =       1.09763341 Ry

----------------------------------------
     Exchange-correlation= SLA  PZ   NOGX NOGC
                           (   1   1   0   0   0   0   0)

----------------------------------------
     number of k points=    16
                       cart. coord. in units 2pi/alat
        k(    1) = (   0.0000000   0.0000000   0.0000000), wk =   0.0092593
        k(    2) = (  -0.1666667   0.1666667  -0.1666667), wk =   0.0740741
        k(    3) = (  -0.3333333   0.3333333  -0.3333333), wk =   0.0740741
        k(    4) = (   0.5000000  -0.5000000   0.5000000), wk =   0.0370370
        k(    5) = (   0.0000000   0.3333333   0.0000000), wk =   0.0555556
        k(    6) = (  -0.1666667   0.5000000  -0.1666667), wk =   0.2222222
